# 02 — Model comparison

Load DIC, LOO, and WAIC metrics for all fitted HDDM candidates, assess convergence (R-hat), rank models, and identify the winning model.

## Prerequisites

1. Prefer **`hddm_analysis/notebooks/`**; package root is located via `_resolve_hddm_root()`.
2. Per-model metric CSVs must exist under `../models/transition_prob_duration_10000samples/` (produced by **`01_model_estimation.ipynb`** or bundled with this release).
3. **R-hat** requires local `*_combined.nc` files *or* the bundled `rhat_cache.csv`. The public repository ships precomputed group-level tables only.

## Model-selection rules (matches manuscript Methods)

1. **Fit metrics:** DIC, LOO, and WAIC on the deviance scale (lower is better).
2. **Convergence:** maximum R-hat over **group-level** parameters only (rows whose names contain `subj` are excluded). Models with max R-hat > 1.01 are flagged as not converged.
3. **Ranking:** equal-weight average of DIC, LOO, and WAIC ranks (`combined_rank = (rank_dic + rank_loo + rank_waic) / 3`).
4. **Winner:** among converged models, select the smallest `combined_rank`; tie-break on lower DIC.
5. **Manuscript model:** `hddm_va_vat` (group-level `v` and `a` vary by condition; non-decision time `t` is a scalar).



In [1]:
import os
from pathlib import Path
import sys

# Package root via _resolve_hddm_root() (data/trials_hddm_ready.tsv).
def _resolve_hddm_root() -> Path:
    """Locate hddm_analysis/ regardless of notebook cwd."""
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "data" / "trials_hddm_ready.tsv").is_file():
            return candidate
        nested = candidate / "hddm_analysis"
        if (nested / "data" / "trials_hddm_ready.tsv").is_file():
            return nested
    if start.name == "notebooks" and (start.parent / "data").is_dir():
        return start.parent
    raise FileNotFoundError(
        "Could not locate hddm_analysis package root "
        "(expected data/trials_hddm_ready.tsv). "
        f"Started search from: {start}"
    )


ROOT = str(_resolve_hddm_root())
SCRIPTS_DIR = os.path.join(ROOT, "scripts")
if SCRIPTS_DIR not in sys.path:
    sys.path.insert(0, SCRIPTS_DIR)
from nc_io import from_netcdf as load_idata
MODEL_DIR = os.path.join(ROOT, "models", "transition_prob_duration_10000samples")
os.makedirs(MODEL_DIR, exist_ok=True)


In [2]:
# --- Configuration ---
# Set LOAD_BEST_IDATA = True only when a local .nc file is available.
LOAD_BEST_IDATA = False

# Recompute R-hat from local .nc files when present; otherwise use bundled rhat_cache.csv.
RECOMPUTE_RHAT = True

# Manuscript winning model (used for validation messaging; selection follows rules above).
MANUSCRIPT_WINNER = "hddm_va_vat"

RHAT_THRESHOLD = 1.01
RHAT_CACHE_VERSION = 3


In [3]:
import glob
import os
from typing import Optional, Tuple

import arviz as az
import numpy as np
import pandas as pd

model_dir = MODEL_DIR

# Collect metric CSVs; skip smoke-test outputs in _test_runs/.
metric_files = sorted(
    f
    for f in glob.glob(os.path.join(model_dir, "*_dic_loo_waic.csv"))
    if "_test_runs" not in f.replace("\\", "/")
)

if not metric_files:
    raise FileNotFoundError(
        "No *_dic_loo_waic.csv files found. "
        f"Run 01_model_estimation.ipynb or use the bundled CSVs in: {model_dir}"
    )

rows = []
for f in metric_files:
    df = pd.read_csv(f)
    if df.empty:
        continue

    row = df.iloc[0].to_dict()
    row["metric_file"] = f

    # Naming convention: hddm_xxx_dic_loo_waic.csv <-> hddm_xxx_combined.nc
    base = os.path.basename(f).replace("_dic_loo_waic.csv", "")
    row["model_name"] = row.get("model_name", base)
    row["idata_file"] = os.path.join(model_dir, f"{base}_combined.nc")
    rows.append(row)

model_table = pd.DataFrame(rows)
if model_table.empty:
    raise ValueError("Metric files were found but all are empty.")

# Harmonise LOO / WAIC column names
for legacy, modern in (("loo_ic", "loo"), ("waic_ic", "waic"), ("elpd_loo", "loo"), ("elpd_waic", "waic")):
    if modern not in model_table.columns and legacy in model_table.columns:
        model_table[modern] = model_table[legacy]

for col in ["dic", "loo", "waic"]:
    if col in model_table.columns:
        model_table[col] = pd.to_numeric(model_table[col], errors="coerce")


def _max_rhat_group_level(idata_file: str) -> Tuple[str, float, Optional[str], float]:
    """Return (idata_file, max_rhat, warning, mtime). warning=None means no issue."""
    max_rhat = np.nan
    warn = None

    if not os.path.exists(idata_file):
        return idata_file, max_rhat, warn, np.nan

    mtime = os.path.getmtime(idata_file)

    try:
        idata = load_idata(idata_file)

        # Match az.summary(idata) row filtering: exclude parameter names containing 'subj'.
        try:
            summary = az.summary(idata, kind="diagnostics")
        except (TypeError, ValueError):
            summary = az.summary(idata)

        idx = summary.index
        if isinstance(idx, pd.MultiIndex):
            row_labels = pd.Series([".".join(str(x) for x in t) for t in idx], index=idx)
        else:
            row_labels = pd.Series([str(x) for x in idx], index=idx)

        keep = ~row_labels.str.lower().str.contains("subj", na=False, regex=False)
        rhat_col = "r_hat" if "r_hat" in summary.columns else ("rhat" if "rhat" in summary.columns else None)
        if rhat_col is None:
            return idata_file, max_rhat, f"Warning: no r_hat column in az.summary for {idata_file}; skipping.", mtime

        if not keep.any():
            return idata_file, max_rhat, f"Warning: no group-level rows left for R-hat in {idata_file}; skipping.", mtime

        sub = pd.to_numeric(summary.loc[keep, rhat_col], errors="coerce").to_numpy()
        sub = sub[np.isfinite(sub)]
        if sub.size > 0:
            max_rhat = float(np.max(sub))

    except Exception as exc:
        warn = f"Failed to compute R-hat for {idata_file}: {exc}"

    return idata_file, max_rhat, warn, mtime


def _nc_basename(idata_file: str) -> str:
    return os.path.basename(idata_file)


def _rel_idata_path(idata_file: str) -> str:
    try:
        return os.path.relpath(idata_file, ROOT).replace("\\", "/")
    except ValueError:
        return _nc_basename(idata_file)


# --- R-hat: recompute from local .nc or fall back to bundled cache ---
cache_path = os.path.join(model_dir, "rhat_cache.csv")


def _load_rhat_cache(path: str) -> dict:
    """Load finite max_rhat values from cache (ignore blank/NaN rows)."""
    out = {}
    if not os.path.exists(path):
        return out
    try:
        cache = pd.read_csv(path)
        required_cols = {"idata_file", "max_rhat", "cache_version"}
        if not required_cols.issubset(cache.columns):
            return out
        cache_ok = cache[cache["cache_version"] == RHAT_CACHE_VERSION]
        for rec in cache_ok.to_dict("records"):
            val = pd.to_numeric(rec.get("max_rhat"), errors="coerce")
            if np.isfinite(val):
                out[_nc_basename(rec["idata_file"])] = float(val)
    except Exception:
        pass
    return out


cache_by_base = _load_rhat_cache(cache_path)
rhat_by_base = dict(cache_by_base)
warn_msgs = []
nc_present = any(os.path.exists(p) for p in model_table["idata_file"])

if RECOMPUTE_RHAT and nc_present:
    for p in model_table["idata_file"].tolist():
        path, val, w, _mtime = _max_rhat_group_level(p)
        base = _nc_basename(path)
        if np.isfinite(val):
            rhat_by_base[base] = float(val)
        elif base not in rhat_by_base:
            rhat_by_base[base] = np.nan
        if w:
            warn_msgs.append(w)

    cache_rows = []
    for p in model_table["idata_file"].tolist():
        base = _nc_basename(p)
        val = rhat_by_base.get(base, np.nan)
        if not np.isfinite(val) and base in cache_by_base:
            val = cache_by_base[base]
            rhat_by_base[base] = val
        cache_rows.append(
            {
                "idata_file": _rel_idata_path(p),
                "max_rhat": val,
                "cache_version": RHAT_CACHE_VERSION,
            }
        )

    if any(np.isfinite(r["max_rhat"]) for r in cache_rows):
        pd.DataFrame(cache_rows).to_csv(cache_path, index=False, encoding="utf-8-sig")
    elif cache_by_base:
        print(
            "Note: R-hat recompute returned no finite values; keeping bundled rhat_cache.csv."
        )
    else:
        print(
            "Note: R-hat recompute returned no finite values and no valid cache is available."
        )
elif cache_by_base:
    if not nc_present:
        print(
            "Note: no local *_combined.nc files found; using bundled rhat_cache.csv for group-level R-hat."
        )
else:
    print(
        "Note: R-hat unavailable (no .nc files and no valid rhat_cache.csv). "
        "DIC/LOO/WAIC ranking still proceeds; convergence flags will be NaN."
    )

for w in warn_msgs:
    print(w)

model_table["max_rhat"] = model_table["idata_file"].map(lambda p: rhat_by_base.get(_nc_basename(p), np.nan))
model_table["rhat_has_gt_1_01"] = model_table["max_rhat"] > RHAT_THRESHOLD
model_table["rhat_converged"] = (~model_table["rhat_has_gt_1_01"]) & model_table["max_rhat"].notna()

# Rank models (lower is better)
model_table["rank_dic"] = model_table["dic"].rank(method="min") if "dic" in model_table.columns else np.nan
model_table["rank_loo"] = model_table["loo"].rank(method="min") if "loo" in model_table.columns else np.nan
model_table["rank_waic"] = model_table["waic"].rank(method="min") if "waic" in model_table.columns else np.nan
rank_parts = [model_table["rank_dic"]]
if "rank_loo" in model_table.columns:
    rank_parts.append(model_table["rank_loo"])
if "rank_waic" in model_table.columns:
    rank_parts.append(model_table["rank_waic"])
model_table["combined_rank"] = sum(rank_parts) / float(len(rank_parts))

sort_cols = ["combined_rank", "dic"]
if "loo" in model_table.columns:
    sort_cols.append("loo")
    if "waic" in model_table.columns:
        sort_cols.append("waic")
model_table = model_table.sort_values(sort_cols, na_position="last").reset_index(drop=True)

print("\n=== Model comparison (sorted by combined rank) ===")
display(model_table)

# Winner: smallest combined_rank among converged models (max R-hat <= 1.01)
converged_table = model_table[model_table["rhat_converged"]].copy()
if len(converged_table) > 0:
    best = (
        converged_table.sort_values(["combined_rank", "dic"], na_position="last")
        .iloc[0]
    )
else:
    print(
        "\nWarning: no converged model available (R-hat missing or all > 1.01); "
        "falling back to best overall combined rank."
    )
    best = model_table.sort_values(["combined_rank", "dic"], na_position="last").iloc[0]

print("\n=== Selected model ===")
print(
    best[
        [
            "model_name",
            "dic",
            "loo",
            "waic",
            "max_rhat",
            "rhat_has_gt_1_01",
            "rhat_converged",
            "combined_rank",
            "idata_file",
            "metric_file",
        ]
    ]
)

if best["model_name"] != MANUSCRIPT_WINNER:
    print(
        f"\nNote: automated selection chose '{best['model_name']}' but the manuscript reports '{MANUSCRIPT_WINNER}'. "
        "Verify that bundled CSVs and R-hat cache match the analysis environment used for the paper."
    )
else:
    print(f"\nSelection matches manuscript winning model: {MANUSCRIPT_WINNER}")

out_csv = os.path.join(model_dir, "model_selection.csv")
model_table.to_csv(out_csv, index=False, encoding="utf-8-sig")
print(f"\nSaved model-selection table: {out_csv}")

best_idata = None
if LOAD_BEST_IDATA:
    if os.path.exists(best["idata_file"]):
        best_idata = load_idata(best["idata_file"])
        print(f"\nLoaded InferenceData for selected model: {best['idata_file']}")
    else:
        print(f"\nInferenceData not found: {best['idata_file']}")
        print("Place a local .nc file or refit with 01_model_estimation.ipynb.")
else:
    print(
        "\nLOAD_BEST_IDATA is False (default). "
    )
